In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

def geocode_csv(input_path, output_path, address_col):
    """
    CSV 파일의 주소 컬럼을 읽어 지오코딩을 수행하고 위도, 경도를 추가하여 저장합니다.
    
    :param input_path: 원본 CSV 파일 경로
    :param output_path: 저정할 CSV 파일 경로
    :param address_col: 주소 데이터가 포함된 컬럼명
    """
    # 1. CSV 파일 불러오기
    try:
        df = pd.read_csv(input_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(input_path, encoding='cp949') # 인코딩 에러 발생 시 변경

    # 2. 지오코더 및 RateLimiter 설정
    # user_agent는 임의의 고유 명칭을 입력해야 에러가 발생하지 않습니다.
    geolocator = Nominatim(user_agent="local_geocoder_service")
    
    # Nominatim의 무료 이용 약관에 따라 요청 간격을 1초로 제한합니다.
    geocode_service = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    print(f"지오코딩을 시작합니다. 총 데이터 수: {len(df)}건")

    # 3. 주소 -> 좌표 변환 수행
    # 변환 실패 시 None 처리를 위해 별도의 함수를 사용합니다.
    def get_coordinates(address):
        if pd.isna(address):
            return None, None
        try:
            location = geocode_service(address)
            if location:
                return location.latitude, location.longitude
            return None, None
        except Exception as e:
            print(f"에러 발생 ({address}): {e}")
            return None, None

    # 위도와 경도 컬럼 생성
    coordinates = df[address_col].apply(get_coordinates)
    df['위도'] = [coords[0] for coords in coordinates]
    df['경도'] = [coords[1] for coords in coordinates]

    # 4. 결과 저장 (Excel 등에서 깨짐 방지를 위해 utf-8-sig 사용)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"지오코딩이 완료되었습니다. 결과 저장 완료: {output_path}")

# 실행 예시
if __name__ == "__main__":
    # 파일 경로 및 주소 컬럼명을 환경에 맞게 수정하여 사용하세요.
    INPUT_CSV = "data/eda/전국공장등록현황.csv"
    OUTPUT_CSV = "data/eda/전국공장등록현황_경위도.csv"
    ADDRESS_COLUMN = "공장주소"
    
    geocode_csv(INPUT_CSV, OUTPUT_CSV, ADDRESS_COLUMN)

지오코딩을 시작합니다. 총 데이터 수: 217048건


KeyboardInterrupt: 